# Math Operations in PyTorch

Every neural network is just math on tensors. This notebook covers each operation:
- the actual math formula
- the PyTorch code
- where it appears in ML

In [ ]:
import torch
import torch.nn.functional as F

## 1. Element-wise Operations

**Math:** apply operation to each corresponding element independently

```
A = [[1, 2],    B = [[5, 6],
     [3, 4]]         [7, 8]]
A * B = [[1×5, 2×6], = [[5, 12],
          [3×7, 4×8]]    [21, 32]]
```

**ML use:** activation functions (ReLU, sigmoid), applying masks, loss scaling

In [ ]:
a = torch.tensor([[1., 2.], [3., 4.]])
b = torch.tensor([[5., 6.], [7., 8.]])

print('a + b:\n', a + b)
print('a - b:\n', a - b)
print('a * b:\n', a * b)   # element-wise — NOT matrix multiply
print('a / b:\n', a / b)
print('a ** 2:\n', a ** 2)
print('sqrt(a):\n', torch.sqrt(a))

## 2. Dot Product

**Math:** multiply element-wise, then sum

```
a = [1, 2, 3],  b = [4, 5, 6]
a · b = (1×4) + (2×5) + (3×6) = 4 + 10 + 18 = 32
```

**ML use:** similarity between two vectors — used in attention to score query vs key

In [ ]:
a = torch.tensor([1., 2., 3.])
b = torch.tensor([4., 5., 6.])

print('dot product:', torch.dot(a, b))   # 32

# Cosine similarity — dot product of unit vectors
a_unit = a / torch.norm(a)
b_unit = b / torch.norm(b)
print('cosine similarity:', torch.dot(a_unit, b_unit).item())   # 1.0 = identical direction

## 3. Matrix Multiplication (matmul) — the most important operation

**Math:** row of A dot-producted with column of B

```
A = [[1, 2],    B = [[5, 6],      C = A @ B
     [3, 4]]         [7, 8]]

C[0,0] = (1×5) + (2×7) = 19
C[0,1] = (1×6) + (2×8) = 22
C[1,0] = (3×5) + (4×7) = 43
C[1,1] = (3×6) + (4×8) = 50

Rule: A(m×n) @ B(n×p) → C(m×p) — inner dims must match
```

**ML use:** IS the linear layer. `y = xW + b` — the `xW` is matmul.
Also: Q @ K.T in attention = similarity scores between all token pairs

In [ ]:
a = torch.tensor([[1., 2.], [3., 4.]])
b = torch.tensor([[5., 6.], [7., 8.]])

print('A @ B:\n', a @ b)              # matrix multiply
print('torch.matmul:\n', torch.matmul(a, b))  # same thing

# Real use: this IS what nn.Linear does internally
X = torch.randn(32, 784)    # 32 examples, 784 features (MNIST)
W = torch.randn(784, 256)   # weight matrix
b_bias = torch.randn(256)   # bias
y = X @ W + b_bias           # = nn.Linear(784, 256)(X)
print('\nLinear layer output shape:', y.shape)   # [32, 256]

# Batched matmul — operates on a batch of matrices
Q = torch.randn(2, 8, 64)   # [batch, heads, seq_len, d_k] — like in attention
K = torch.randn(2, 8, 64)
scores = Q @ K.transpose(-2, -1)   # [2, 8, 8] — attention scores
print('Attention scores shape:', scores.shape)

## 4. Transpose

**Math:** flip rows and columns

```
A = [[1, 2, 3],     A.T = [[1, 4],
     [4, 5, 6]]             [2, 5],
                             [3, 6]]
A(2×3) → A.T(3×2)
```

**ML use:** needed before matmul when dimensions don't align. In attention: `Q @ K.T`

In [ ]:
a = torch.tensor([[1., 2., 3.], [4., 5., 6.]])  # [2, 3]

print('a.T shape:         ', a.T.shape)              # [3, 2]
print('transpose(0,1):    ', a.transpose(0, 1).shape)

# permute — generalized transpose for 3D+ tensors
img = torch.randn(224, 224, 3)    # H × W × C (OpenCV format)
img = img.permute(2, 0, 1)        # → C × H × W (PyTorch format)
print('image permuted:    ', img.shape)

## 5. Reduction Operations

**Math:** reduce a dimension to a scalar

```
A = [[1, 2, 3],
     [4, 5, 6]]

sum(all)   = 21
sum(dim=0) = [5, 7, 9]   ← collapse rows (sum each column)
sum(dim=1) = [6, 15]     ← collapse cols (sum each row)
```

**ML use:** mean loss over a batch, global average pooling in CNNs, argmax for predictions

In [ ]:
a = torch.tensor([[1., 2., 3.], [4., 5., 6.]])

print('sum all:    ', a.sum())
print('sum dim=0:  ', a.sum(dim=0))    # [5, 7, 9]
print('sum dim=1:  ', a.sum(dim=1))    # [6, 15]
print('mean:       ', a.mean())
print('max:        ', a.max())
print('argmax:     ', a.argmax())       # flat index
print('argmax dim1:', a.argmax(dim=1))  # per-row argmax — used for predicted class

## 6. Broadcasting

**Math:** automatically expand smaller tensor to match larger tensor shape

```
A = [[1, 2, 3],    bias = [10, 20, 30]
     [4, 5, 6]]

A + bias → bias broadcast across rows:
         = [[11, 22, 33],
            [14, 25, 36]]
```

**ML use:** adding bias to every example in a batch — bias is [256], output is [32, 256]

In [ ]:
a    = torch.tensor([[1., 2., 3.], [4., 5., 6.]])  # [2, 3]
bias = torch.tensor([10., 20., 30.])                # [3]

print('a + bias:\n', a + bias)   # bias broadcast to [2, 3]

# Common shape errors
try:
    c = torch.randn(3, 4) + torch.randn(3, 5)   # incompatible
except RuntimeError as e:
    print('\nShape error:', e)

## 7. Softmax

**Math:** converts raw scores (logits) to probabilities that sum to 1

```
x = [2.0, 1.0, 0.1]
exp(x) = [7.39, 2.72, 1.11]   sum = 11.22
softmax = [0.659, 0.242, 0.099]  ← sums to 1.0
```

**ML use:** final layer of classifiers, attention weights in transformers

In [ ]:
logits = torch.tensor([2.0, 1.0, 0.1])
probs  = F.softmax(logits, dim=0)
print('softmax:', probs)
print('sum:    ', probs.sum())   # always 1.0

# Batched — dim=1 applies softmax per example
logits_batch = torch.randn(4, 10)   # 4 examples, 10 classes
probs_batch  = F.softmax(logits_batch, dim=1)
print('\nbatch probs shape:', probs_batch.shape)
print('row sums:         ', probs_batch.sum(dim=1))   # all 1.0

## 8. Norm

**Math:** L2 norm = magnitude of a vector = sqrt(Σ xᵢ²)

```
x = [3, 4]
||x|| = sqrt(3² + 4²) = sqrt(25) = 5
```

**ML use:** gradient clipping (if ||grad|| > threshold, scale it down), weight decay, normalizing embeddings

In [ ]:
x = torch.tensor([3., 4.])
print('L2 norm:', torch.norm(x))        # 5.0
print('L1 norm:', torch.norm(x, p=1))   # |3| + |4| = 7

# Normalize to unit vector (length = 1)
x_unit = x / torch.norm(x)
print('unit vector:', x_unit)
print('unit norm:  ', torch.norm(x_unit))   # 1.0

# Gradient clipping — used in LLM training to prevent exploding gradients
gradients = torch.tensor([100., 200., 50.])
max_norm  = 1.0
grad_norm = torch.norm(gradients)
if grad_norm > max_norm:
    gradients = gradients * (max_norm / grad_norm)
print('\nclipped gradient norm:', torch.norm(gradients).item())